Eric Arnold

ML Midterm

Prof. Khoei

In [1]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.model_selection import train_test_split
from pprint import pprint

from scipy.stats import chi2_contingency
from collections import defaultdict

from sklearn.decomposition import PCA
# from umap import UMAP

## Load datasets

When loading, polars did not automatically recognize any of the dtypes, so I had to manually cast them. Some categories in the Assess portion of the dataset are actually floats, but they were very large, so rounding introduced by casting them as integers is small.

In [2]:
cols = ['ID', 'CENTER', 'FY', 'SIC', 'NAICS', 'STATE', 'SALES', 'EMPLOYEES', 'PLANT_AREA', 'PRODUCTS', 'PRODUNITS', 'PRODLEVEL', 'PRODHOURS', 'RECC_COUNT_recc', 'EC_plant_cost', 'EC_plant_usage', 'ED_plant_cost', 'ED_plant_usage', 'EF_plant_cost', 'E2_plant_cost', 'E2_plant_usage', 'E3_plant_cost', 'E3_plant_usage', 'E4_plant_cost', 'E4_plant_usage', 'E5_plant_cost', 'E5_plant_usage', 'E6_plant_cost', 'E6_plant_usage', 'E7_plant_cost', 'E7_plant_usage', 'E8_plant_cost', 'E8_plant_usage', 'E9_plant_cost', 'E9_plant_usage', 'E10_plant_cost', 'E10_plant_usage', 'E11_plant_cost', 'E11_plant_usage', 'E12_plant_cost', 'E12_plant_usage', 'W0_plant_cost', 'W0_plant_usage', 'W1_plant_cost', 'W1_plant_usage', 'W2_plant_cost', 'W2_plant_usage', 'W3_plant_cost', 'W3_plant_usage', 'W4_plant_cost', 'W4_plant_usage', 'W5_plant_cost', 'W5_plant_usage', 'W6_plant_cost', 'W6_plant_usage']
str_cols = ["ID", "CENTER", "SIC", "NAICS", "STATE", "PRODUCTS"] 
int_cols = [c for c in cols if c not in str_cols]
schema = {c: pl.Utf8 for c in str_cols} | {c: pl.Int64 for c in int_cols}

ASSESS = pl.read_excel("ITAC_Database_20261002.xlsx", sheet_name="ASSESS", schema_overrides=schema)

cols = ['SUPERID', 'ID', 'ARC2', 'IMPSTATUS', 'IMPCOST', 'PSOURCCODE', 'PCONSERVED', 'PSOURCONSV', 'PSAVED', 'SSOURCCODE', 'SCONSERVED', 'SSOURCONSV', 'SSAVED', 'TSOURCCODE', 'TCONSERVED', 'TSOURCONSV', 'TSAVED', 'QSOURCCODE', 'QCONSERVED', 'QSOURCONSV', 'QSAVED', 'REBATE', 'INCREMNTAL', 'FY', 'IC_CAPITAL', 'IC_OTHER', 'PAYBACK', 'BPTOOL']
str_cols = ["SUPERID", "ID", "AR_NUMBER", "IMPSTATUS", "BPTOOL", "PSOURCCODE", "SSOURCCODE", "TSOURCCODE", "QSOURCCODE", "REBATE", "INCREMNTAL"]
float_cols = ["ARC2", "PAYBACK"]
int_cols = ["APPCODE", "AR_NUMBER", "FY", "IMPCOST", "PCONSERVED", "PSOURCONSV", "PSAVED", "SCONSERVED", "SSOURCONSV", "SSAVED", "TCONSERVED", "TSOURCONSV", "TSAVED", "QCONSERVED", "QSOURCONSV", "QSAVED", "IC_CAPITAL", "IC_OTHER"]
schema = ({c: pl.Utf8 for c in str_cols} | {c: pl.Float64 for c in float_cols} | {c: pl.Int64 for c in int_cols})


RECC = pl.read_excel("ITAC_Database_20261002.xlsx", sheet_name="RECC", schema_overrides=schema)

# Problem Statement

The Industrial Training and Assessment Centers (ITAC) organization is a DOE program that assesses industrial facilities, chiefly small and medium-sized factories, and makes recommendations about how to save energy and money on production, roughly speaking. The dataset indicates whether or not the recommended changes were implemented, how much it cost, and how much was saved (in terms of resource units, which might be gallons of fuel, kilowatts of electricity, etc.).

The obvious thing to comb through all the variables and see if any are associated with implementation of recommendations, and subsequent savings.

https://catalog.data.gov/dataset/industrial-assessment-centers-database

In [3]:
ASSESS_ids = ASSESS["ID"].unique()
RECC_ids = RECC["ID"].unique()
null_recc_ids = RECC_ids.filter(~RECC_ids.is_in(ASSESS_ids.implode()))
null_assess_ids = ASSESS_ids.filter(~ASSESS_ids.is_in(RECC_ids.implode()))
print(null_recc_ids)
print(null_assess_ids)
RECC = RECC.filter(~pl.col("ID").is_in(null_recc_ids.implode()))
# fig = go.Figure(go.Histogram(RECC[id]))

shape: (3,)
Series: 'ID' [str]
[
	"AM1008"
	"CM1056"
	"AM1007"
]
shape: (0,)
Series: 'ID' [str]
[
]


# EDA

The data can be divided into three sections:

- Assessment data (ASSESS)
    - Categorical
    - Continuous (integer)
- Plant data (ASSESS)
    - Sparse Continuous (integer)
- Outcome data (RECC)
    - Continuous (integer/float)
    - Bernoulli

Here with categorical assessment data, we can see that assessing universities are unevenly distributed, causing the distribution of the number of plants per state to also be unevenly distributed. For splitting our data into train and validate, it makes sense to split by state since whatever regional affects there are probably follow this label.

In [4]:
cat_cols = ["CENTER", "FY", "STATE"]
titles = ["Assessor", "Fiscal Year", "State"]
subs = make_subplots(2, 3, subplot_titles=titles)

for i, col in enumerate(cat_cols):
    x, y = divmod(i, 3)
    subs.add_trace(go.Histogram(x=ASSESS[col].to_list()), row=x+1, col=y+1)

subs.show()

NAICS codes are 6-digit codes that describe what kind of industry a business is in. The first three digits correspond to the larger industry, while subsequent digits indicate sub-industries. The source for the codes is https://www.census.gov/naics/?48967.

These perform the same role as SIC codes (Standard Industrial Classification) but are a more modern version. However, I found that many entries have null NAICS, whereas there is only 1 for SIC. So we'll have to use both categorizations to cover the whole dataset.

In [21]:
naics3_names = {
    # agriculture, mining, utilities, construction
    "111": "Crop production", "112": "Animal production & aquaculture",
    "113": "Forestry & logging", "114": "Fishing, hunting & trapping",
    "115": "Agriculture & forestry support",
    "211": "Oil & gas extraction", "212": "Mining (except oil & gas)",
    "213": "Mining support activities", "221": "Utilities",
    "236": "Construction of buildings", "237": "Heavy & civil engineering construction",
    "238": "Specialty trade contractors",
    # manufacturing
    "311": "Food", "312": "Beverage & tobacco", "313": "Textile mills",
    "314": "Textile product mills", "315": "Apparel", "316": "Leather & allied",
    "321": "Wood products", "322": "Paper", "323": "Printing",
    "324": "Petroleum & coal", "325": "Chemicals", "326": "Plastics & rubber",
    "327": "Nonmetallic minerals", "331": "Primary metals",
    "332": "Fabricated metals", "333": "Machinery",
    "334": "Computer & electronic", "335": "Electrical equipment",
    "336": "Transportation equipment", "337": "Furniture", "339": "Miscellaneous",
    # wholesale and retail
    "423": "Wholesale, durable goods", "424": "Wholesale, nondurable goods",
    "425": "Wholesale agents & brokers",
    "441": "Motor vehicle & parts dealers", "444": "Building material & garden retailers",
    "445": "Food & beverage retailers", "449": "Furniture, home & electronics retailers",
    "455": "General merchandise retailers", "456": "Health & personal care retailers",
    "457": "Gasoline stations & fuel dealers", "458": "Clothing, shoe & jewelry retailers",
    "459": "Sporting, hobby, book & misc retailers",
    # transportation and warehousing
    "481": "Air transportation", "482": "Rail transportation", "483": "Water transportation",
    "484": "Truck transportation", "485": "Transit & ground passenger transportation",
    "486": "Pipeline transportation", "487": "Scenic & sightseeing transportation",
    "488": "Transportation support activities", "491": "Postal service",
    "492": "Couriers & messengers", "493": "Warehousing & storage",
    # information, finance, real estate
    "512": "Motion picture & sound recording", "513": "Publishing",
    "516": "Broadcasting & content providers", "517": "Telecommunications",
    "518": "Computing infrastructure & data processing", "519": "Other information services",
    "521": "Monetary authorities", "522": "Credit intermediation",
    "523": "Securities & investments", "524": "Insurance carriers", "525": "Funds & trusts",
    "531": "Real estate", "532": "Rental & leasing", "533": "Lessors of intangible assets",
    # services
    "541": "Professional, scientific & technical services", "551": "Management of companies",
    "561": "Administrative & support services", "562": "Waste management & remediation",
    "611": "Educational services", "621": "Ambulatory health care", "622": "Hospitals",
    "623": "Nursing & residential care", "624": "Social assistance",
    "711": "Performing arts & spectator sports", "712": "Museums & historical sites",
    "713": "Amusement, gambling & recreation", "721": "Accommodation",
    "722": "Food services & drinking places",
    "811": "Repair & maintenance", "812": "Personal & laundry services",
    "813": "Religious, civic & professional organizations", "814": "Private households",
    # public administration
    "921": "General government support", "922": "Justice, public order & safety",
    "923": "Human resource program administration", "924": "Environmental program administration",
    "925": "Housing & urban development administration", "926": "Economic program administration",
    "927": "Space research & technology", "928": "National security & international affairs",
}

sector_names = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities", "23": "Construction",
    "31": "Manufacturing", "32": "Manufacturing", "33": "Manufacturing",
    "42": "Wholesale Trade", "44": "Retail Trade", "45": "Retail Trade",
    "48": "Transportation and Warehousing", "49": "Transportation and Warehousing",
    "51": "Information", "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional, Scientific, and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Support and Waste Management and Remediation Services",
    "61": "Educational Services", "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation", "72": "Accommodation and Food Services",
    "81": "Other Services (except Public Administration)", "92": "Public Administration",
}

ASSESS = ASSESS.with_columns(NAICS2=pl.col("NAICS").str.slice(0, 2),NAICS3=pl.col("NAICS").str.slice(0, 3),
).with_columns(NAICS2_name=pl.col("NAICS2").replace_strict(sector_names, default=None, return_dtype=pl.Utf8),
NAICS3_name=pl.col("NAICS3").replace_strict(naics3_names, default=None, return_dtype=pl.Utf8),
)

sic2_names = {
    "01": "Agricultural production, crops", "02": "Agricultural production, livestock",
    "07": "Agricultural services", "08": "Forestry", "09": "Fishing, hunting & trapping",
    "10": "Metal mining", "12": "Coal mining", "13": "Oil & gas extraction",
    "14": "Nonmetallic minerals mining", "15": "Building construction",
    "16": "Heavy construction", "17": "Special trade contractors",
    "20": "Food & kindred products", "21": "Tobacco products", "22": "Textile mill products",
    "23": "Apparel", "24": "Lumber & wood products", "25": "Furniture & fixtures",
    "26": "Paper & allied products", "27": "Printing & publishing", "28": "Chemicals & allied products",
    "29": "Petroleum refining", "30": "Rubber & misc plastics", "31": "Leather products",
    "32": "Stone, clay, glass & concrete", "33": "Primary metals", "34": "Fabricated metals",
    "35": "Industrial machinery & equipment", "36": "Electronic & electrical equipment",
    "37": "Transportation equipment", "38": "Instruments & related products",
    "39": "Miscellaneous manufacturing",
    "40": "Railroad transportation", "41": "Local passenger transit", "42": "Trucking & warehousing",
    "43": "Postal service", "44": "Water transportation", "45": "Air transportation",
    "46": "Pipelines", "47": "Transportation services", "48": "Communications",
    "49": "Electric, gas & sanitary services",
    "50": "Wholesale, durable goods", "51": "Wholesale, nondurable goods",
    "52": "Building materials & garden retail", "53": "General merchandise retail",
    "54": "Food stores", "55": "Auto dealers & gas stations", "56": "Apparel retail",
    "57": "Furniture & home furnishings retail", "58": "Eating & drinking places", "59": "Misc retail",
    "60": "Depository institutions", "61": "Nondepository credit", "62": "Security & commodity brokers",
    "63": "Insurance carriers", "64": "Insurance agents", "65": "Real estate", "67": "Holding & investment offices",
    "70": "Hotels & lodging", "72": "Personal services", "73": "Business services",
    "75": "Auto repair & parking", "76": "Misc repair services", "78": "Motion pictures",
    "79": "Amusement & recreation", "80": "Health services", "81": "Legal services",
    "82": "Educational services", "83": "Social services", "84": "Museums & gardens",
    "86": "Membership organizations", "87": "Engineering & management services",
    "88": "Private households", "89": "Misc services",
    "91": "Executive, legislative & general government", "92": "Justice, public order & safety",
    "93": "Public finance & taxation", "94": "Human resource administration",
    "95": "Environmental quality & housing", "96": "Economic programs administration",
    "97": "National security & international affairs", "99": "Nonclassifiable establishments",
}

ASSESS = ASSESS.with_columns(SIC2=pl.col("SIC").str.zfill(4).str.slice(0, 2),SIC3=pl.col("SIC").str.zfill(4).str.slice(0, 3),
).with_columns(SIC2_name=pl.col("SIC2").replace_strict(sic2_names, default=None, return_dtype=pl.Utf8))


print("SIC nulls:", ASSESS["SIC"].null_count(), "NAICS nulls:", ASSESS["NAICS"].null_count())

SIC nulls: 1 NAICS nulls: 10970


Per the plot below, we can see that 

In [24]:
sic = (ASSESS.drop_nulls("SIC2").group_by("SIC2", "SIC2_name").len().sort("len", descending=True).with_columns(label=pl.col("SIC2") + " " + pl.col("SIC2_name").fill_null("(unmapped)")))
sec = (ASSESS.drop_nulls("NAICS2").group_by("NAICS2", "NAICS2_name").len().sort("len", descending=True).with_columns(label=pl.col("NAICS2") + " " + pl.col("NAICS2_name").fill_null("(unmapped)")))
sub = (ASSESS.drop_nulls("NAICS3").group_by("NAICS3", "NAICS3_name").len().sort("len", descending=True).with_columns(label=pl.col("NAICS3") + " " + pl.col("NAICS3_name").fill_null("(unmapped)")))

subs = make_subplots(1, 3, subplot_titles=["SIC Major Group (2-digit)", "NAICS Sector (2-digit)", "NAICS Subsector (3-digit)"], horizontal_spacing=0.2)

subs.add_trace(go.Bar(x=sic["len"].to_list(), y=sic["label"].to_list(), orientation="h"), 1, 1)
subs.add_trace(go.Bar(x=sec["len"].to_list(), y=sec["label"].to_list(), orientation="h"), 1, 2)
subs.add_trace(go.Bar(x=sub["len"].to_list(), y=sub["label"].to_list(), orientation="h"), 1, 3)

subs.update_yaxes(type="category", autorange="reversed")   # most common at top
subs.update_xaxes(title_text="Assessments")
subs.update_layout(height=1024, width=1280, showlegend=False, title_text="Assessments by SIC and NAICS Classification")
subs.show()

In [6]:
assessments_per = RECC["ID"].value_counts()["count"].to_list()
implementations_per = RECC.filter(pl.col("IMPSTATUS") == "I")["ID"].value_counts()["count"].to_list()
negs_per = RECC.filter(pl.col("IMPSTATUS") == "N")["ID"].value_counts()["count"].to_list()
state_df = RECC.select("ID", "IMPSTATUS").join(ASSESS.select("ID", "STATE"), on="ID", how="left")
state_df = state_df.group_by("ID", "IMPSTATUS", "STATE").agg(pl.len()).sort("len")
state_assess = state_df.group_by("STATE").agg(pl.col("len").sum()).sort("len")
state_means = state_df.group_by("STATE").agg(pl.col("len").filter(pl.col("IMPSTATUS") == "I").sum() / pl.col("len").sum()).sort(by="len", descending=True)
subs = make_subplots(3, 2, subplot_titles=["Assessments per plant", "Implementations per plant", "Negatives per plant", "Assessments per state", "Mean Implementations/Assessments per plant per state"])

subs.add_trace(go.Histogram(x=assessments_per), 1, 1)
subs.add_trace(go.Histogram(x=implementations_per), 1, 2)
subs.add_trace(go.Histogram(x=negs_per), 2, 1)
subs.add_trace(go.Bar(x=state_assess["STATE"], y=state_assess["len"]), 2, 2)
subs.add_trace(go.Bar(x=state_means["STATE"], y=state_means["len"]), 3, 1)

subs.update_xaxes(title_text="Assessments", row=1, col=1)
subs.update_yaxes(title_text="Plants", row=1, col=1)
subs.update_xaxes(title_text="Implementations", row=1, col=2)
subs.update_yaxes(title_text="Plants", row=1, col=2)
subs.update_xaxes(title_text="Negatives", row=2, col=1)
subs.update_yaxes(title_text="Plants", row=2, col=1)
subs.update_xaxes(title_text="State", row=2, col=2)
subs.update_yaxes(title_text="Assessments", row=2, col=2)
subs.update_xaxes(title_text="State", row=3, col=1)
subs.update_yaxes(title_text="Mean Value", row=3, col=1)
subs.update_layout(height=768, width=1024)

subs.show()

The distribution of implementations per plant is slightly right-shifted compared to negatives per plant, indicating that recommendations are implemented more often than not. There is a huge amount of variability in terms of the ratio of successful implementations to total implementations per state, and in the number of assessments per state. For splitting the dataset into train/val, we probably want to balance our subset by state.

My strategy here is to select the ids and then apply them to the dataframes before I use them, instead of selecting rows.

In [7]:
ids = ASSESS.select("ID", "STATE").unique(subset="ID", keep="first", maintain_order=True)
train_ids, val_ids = train_test_split(ids, test_size=0.2, random_state=0, stratify=ids["STATE"])

## Production Unit (produnit) analysis

IAC defines the unit type for measuring the quantity of a product produced using the following codes:

- 0: N/A
- 1: Pieces
- 2: Pounds
- 3: Tons
- 4: BBL (beer barrel?)
- 5: Thousands of Gallons
- 6: Thousands of Feet or Square Feet
- 7: Bushels

I wanted to verify that there was no significant difference between produnit proportions between train and val as a proxy for "the kind of thing produced." This is because my split did not take into account any information about what kinds of products were produced. Since states might vary greatly in terms of industry type, I wanted to test for this. There were no significant differences subdividing by state or overall.

In [8]:
pu_train = ASSESS.select("ID", "PRODUNITS", "STATE").filter(pl.col("ID").is_in(train_ids["ID"].implode()))
pu_train = pu_train.with_columns(pl.col("PRODUNITS").fill_null(0))
pu_state_train = pu_train.group_by("STATE", "PRODUNITS").agg(pl.len()).sort("STATE", "PRODUNITS")

pu_val = ASSESS.select("ID", "PRODUNITS", "STATE").filter(pl.col("ID").is_in(val_ids["ID"].implode()))
pu_val = pu_val.with_columns(pl.col("PRODUNITS").fill_null(0))
pu_state_val = pu_val.group_by("STATE", "PRODUNITS").agg(pl.len()).sort("STATE", "PRODUNITS")

pu_train = pu_train.with_columns((pl.col("STATE").cast(pl.String) + "_" + pl.col("PRODUNITS").cast(pl.String)).alias("sp"))
pu_val = pu_val.with_columns((pl.col("STATE").cast(pl.String) + "_" + pl.col("PRODUNITS").cast(pl.String)).alias("sp"))

print("Testing STATE x PRODUNITS x Split")
tbl = (
    pl.concat([
        pu_train.group_by("sp").agg(pl.len().alias("n")).with_columns(pl.lit("train").alias("split")),
        pu_val.group_by("sp").agg(pl.len().alias("n")).with_columns(pl.lit("val").alias("split")),
    ])
    .pivot(on="sp", index="split", values="n")
    .fill_null(0)
    .sort("split")
)
chi2, p, dof, expected = chi2_contingency(tbl.drop("split").to_numpy())
print("test_stat:", chi2, "p:", p, "dof:", dof)

print("\nTesting PRODUNITS x Split")
both = pl.concat([pu_state_train.with_columns(pl.lit("train").alias("split")),
                  pu_state_val.with_columns(pl.lit("val").alias("split"))])
overall = (
    both.group_by("split", "PRODUNITS").agg(pl.col("len").sum())
    .pivot(on="PRODUNITS", index="split", values="len")
    .fill_null(0)
    .sort("split")
)
chi2, p, dof, expected = chi2_contingency(overall.drop("split").to_numpy())
print("test_stat:", chi2, "p:", p, "dof:", dof)

Testing STATE x PRODUNITS x Split
test_stat: 289.4479904158215 p: 0.9737593880676818 dof: 338

Testing PRODUNITS x Split
test_stat: 1.665458525527658 p: 0.9760229667732913 dof: 7


Now that the dataset is split, we can do more EDA on train only.

Here we can examine the relationships between the assessment variables. Since these are upstream from the recommendation, we can treat them as indepdent variables. I'm analyzing these separately from the resource utilization variables since they're relatively dense.

Log transforming the distributions for sales, employees, plant area, prod level, and recc count normalizes these distributions.

Nulls in employees, sales, and prod level seem like good candidates for interpolation within their respective categories (state, produnit). About 1/4 of the data lacks a plant area measurement, and I'd expect that this is associated with agriculture or some industry where this is hard to quanitfy.

For prod hours (annual), a large number of entries have 8600-8700 hours, which across 365 days reflects 24 hours a day of production. I think that anything more than 24/hours a day doesn't make sense, so we can cap these at the maximum possible value. For interpolation, we can probably take the mean of its associate category, depending on how prodhours is assocaited with other variables.

In [9]:
ASSESS_train = ASSESS.filter(pl.col("ID").is_in(train_ids["ID"].implode()))
a_cols = ['SALES', 'EMPLOYEES', 'PLANT_AREA', 'PRODLEVEL', 'PRODHOURS', 'RECC_COUNT_recc']
nulls = {cat: ASSESS_train[cat].is_null().sum() + (ASSESS_train[cat] == 0).sum() for cat in a_cols}
pprint(nulls)

sales_trf = ASSESS_train["SALES"].log().to_numpy()
empl_trf = ASSESS_train["EMPLOYEES"].log().to_numpy()
area_trf = ASSESS_train["PLANT_AREA"].log().to_numpy()
lvl_trf = ASSESS_train["PRODLEVEL"].log().to_numpy()
hrs_trf = (ASSESS_train.with_columns(pl.col("PRODHOURS").clip(upper_bound=8760))["PRODHOURS"] / 365).to_numpy()
recc_trf = ASSESS_train["RECC_COUNT_recc"].log().to_numpy()

subs = make_subplots(2, 3, subplot_titles=[f"Distribution of {cat}" for cat in a_cols])
subs.add_trace(go.Histogram(x=sales_trf), 1, 1)
subs.add_trace(go.Histogram(x=empl_trf), 1, 2)
subs.add_trace(go.Histogram(x=area_trf), 1, 3)
subs.add_trace(go.Histogram(x=lvl_trf), 2, 1)
subs.add_trace(go.Histogram(x=hrs_trf), 2, 2)
subs.add_trace(go.Histogram(x=recc_trf), 2, 3)

subs.update_xaxes(title_text="Sales $$ (log scale)", row=1, col=1)
subs.update_yaxes(title_text="Plants", row=1, col=1)
subs.update_xaxes(title_text="Employees (log scale)", row=1, col=2)
subs.update_yaxes(title_text="Plants", row=1, col=2)
subs.update_xaxes(title_text="Plant Area  (log scale)", row=1, col=3)
subs.update_yaxes(title_text="Plants", row=1, col=3)
subs.update_xaxes(title_text="Production Level (log scale)", row=2, col=1)
subs.update_yaxes(title_text="Plants", row=2, col=1)
subs.update_xaxes(title_text="Production Hours / Day", row=2, col=2)
subs.update_yaxes(title_text="Plants", row=2, col=2)
subs.update_xaxes(title_text="Recommendation Count (log scale)", row=2, col=3)
subs.update_yaxes(title_text="Plants", row=2, col=3)

subs.update_layout(height=768, width=1024, showlegend=False)

subs.show()

{'EMPLOYEES': 14,
 'PLANT_AREA': 3263,
 'PRODHOURS': 67,
 'PRODLEVEL': 1009,
 'RECC_COUNT_recc': 0,
 'SALES': 104}


## Correlation Matrix

I chose to use Pearson here because the transformations were mostly successful at normalizing the data, and it is more powerful than Spearman. A large enough amount of the data is non-null for inference on these variables to be robust.

Correlations between the variables are unexpectedly low, which throws a wrench into the interpolation method. Interestingly there is no correlation with year (FY) and sales, which I expected to be related due to inflation, and the fact that the economy is continuously "growing."

In [10]:
corr = pl.DataFrame({"SALES": sales_trf, "EMPLOYEES": empl_trf, "PLANT_AREA": area_trf, "PRODLEVEL": lvl_trf, "PRODHOURS": hrs_trf, "RECC_COUNT_recc": recc_trf, "FY": ASSESS_train["FY"]
}).with_columns(pl.when(pl.all().is_finite()).then(pl.all()).otherwise(None))
print("shape before dropping nulls", corr.shape)
corr_full = corr.drop_nulls()
print("after", corr_full.shape)

corells = corr_full.corr()
fig = go.Figure(go.Heatmap(z = corells.to_numpy(), x = corells.columns, y = corells.columns, zmin=-1, zmax=1, text=np.round(corells.to_numpy(), 2), texttemplate="%{text}"))
fig.update_layout(height=768, width=768, title_text="Correlation Matrix Dense Assessment Variables")
fig.show()

shape before dropping nulls (18405, 7)
after (14511, 7)


## Sparse Assessment Data

This category of the dataset deals with resource consumption, such as electrical consumption, steam consumption, natural gas consumption, etc., and waste generation. I approached this set of variables separately because most plants contain only a subset of resources utilized and wastes generated. My plan for these variables is to treat zeros as zeros and use either PCA or UMAP to do dimensionality reduction. I expect UMAP to handle sparsity better since it uses euclidean distance and does not consider directions of variance individually (as a linear combination), but I'll use PCA to see about how many dimensions reside in the data.

Correlation analysis shows that only strong relationships exist between usages and costs. Electricity and natural gas are somewhat correlated. Costs between resources are somewhat correlated I would suspect because regional variation. Electrical demand usage and electrical usage are not correlated, which is interesting. Electrical demand deals with power supplied in a timely fashion, e.g. for a particular machine, whereas electrical usage characterizes steady-state power consumption. This can be thought of as the difference between streamed tokens versus batched ones -- the former costs more because it's served in real time.

Waste has hardly any correlations at all. The absence of linear relationships across the data mean that PCA will not perform well.

In [11]:
energies = ['EC_plant_cost', 'EC_plant_usage', 'ED_plant_cost', 'ED_plant_usage', 'EF_plant_cost', 'E2_plant_cost', 'E2_plant_usage', 'E3_plant_cost', 'E3_plant_usage', 'E4_plant_cost', 'E4_plant_usage', 'E5_plant_cost', 'E5_plant_usage', 'E6_plant_cost', 'E6_plant_usage', 'E7_plant_cost', 'E7_plant_usage', 'E8_plant_cost', 'E8_plant_usage', 'E9_plant_cost', 'E9_plant_usage', 'E10_plant_cost', 'E10_plant_usage', 'E11_plant_cost', 'E11_plant_usage', 'E12_plant_cost', 'E12_plant_usage'] 
wastes = ['W0_plant_cost', 'W0_plant_usage', 'W1_plant_cost', 'W1_plant_usage', 'W2_plant_cost', 'W2_plant_usage', 'W3_plant_cost', 'W3_plant_usage', 'W4_plant_cost', 'W4_plant_usage', 'W5_plant_cost', 'W5_plant_usage', 'W6_plant_cost', 'W6_plant_usage']

e_labels = {
    "EC": "Electricity", "ED": "Electricity demand", "EF": "Electricity fees",
    "E2": "Natural gas", "E3": "LPG", "E4": "#1 Fuel oil", "E5": "#2 Fuel oil",
    "E6": "#4 Fuel oil", "E7": "#6 Fuel oil", "E8": "Coal", "E9": "Wood",
    "E10": "Paper", "E11": "Other gas", "E12": "Other energy",
}
w_labels = {
    "W0": "Water consumption", "W1": "Water disposal",
    "W2": "Liquid waste (non-haz)", "W3": "Liquid waste (haz)",
    "W4": "Solid waste (non-haz)", "W5": "Solid waste (haz)",
    "W6": "Gaseous waste",
}

def rename_map(cols, labels):
    # "E2_plant_cost" -> "Natural gas cost"
    return {c: f"{labels[c.split('_')[0]]} {c.split('_')[2]}" for c in cols}

E_df = ASSESS_train.select(energies).fill_null(0).rename(rename_map(energies, e_labels))
W_df = ASSESS_train.select(wastes).fill_null(0).rename(rename_map(wastes, w_labels))

E_corr = E_df.corr()
W_corr = W_df.corr()

subs = make_subplots(2, 1, subplot_titles=["Energy Correlations", "Waste Correlations"])

subs.add_trace(go.Heatmap(z=E_corr.to_numpy(), x=E_corr.columns, y=E_corr.columns, zmin=-1, zmax=1, text=np.round(E_corr.to_numpy(), 2), texttemplate="%{text}", showscale=False), 1, 1)
subs.add_trace(go.Heatmap(z=W_corr.to_numpy(), x=W_corr.columns, y=W_corr.columns, zmin=-1, zmax=1, text=np.round(W_corr.to_numpy(), 2), texttemplate="%{text}"), 2, 1)

subs.update_layout(height=1536, width=1280, title_text="Correlation Matrices Dense Assessment Variables")
subs.show()



### Dimensionality reduction

Since these variables are uncorrelated, it makes the most sense to use all of them and see what happens.

In [12]:
def pca_umap_analysis(df, c=5):
    print("==pca==")
    pca = PCA(n_components=c)
    pca_emb = pca.fit_transform(df.to_numpy())
    print(pca.explained_variance_)

    print("\n==umap==")
    


    